# Two-Stage Leakage-Free Band Selection Pipeline\n",
    "## All 10 band-based EEG sheets\n",
    "\n",
    "Implements the methodology from the 3_* reference notebooks, extended to cover\n",
    "**all 10 band-based sheets** in the patient xlsx files.\n",
    "\n",
    "Per-Hz sheets (`FFT_abs_1to50Hz_uV2`, etc.) are excluded — too many bins for\n",
    "a ~100-patient cohort.\n",
    "\n",
    "---\n",
    "\n",
    "## Stage 1 — Band selection via win-counting (training data only)\n",
    "For each outer fold, the training patients are split into `INNER_N_SPLITS` inner folds.\n",
    "In every inner fold **each candidate band is evaluated individually** with a\n",
    "LogReg + StandardScaler pipeline (mean + std across channels = 2 features/band).\n",
    "The band that achieves the highest balanced accuracy in that inner fold wins one point.\n",
    "After all inner folds the top-`N_TOP_BANDS` bands (most wins, ties broken by\n",
    "average inner BA) are selected.  All selection decisions are made exclusively from\n",
    "**training** samples — the test fold is never seen.\n",
    "\n",
    "## Stage 2 — Grid search with selected bands\n",
    "A full LogReg hyperparameter grid (solvers × penalties × C values) is searched via\n",
    "`GridSearchCV` using the same inner-fold scheme, combining the selected bands\n",
    "(4 features total for N_TOP_BANDS=2).  The best estimator evaluates on the\n",
    "held-out outer test fold.\n",
    "\n",
    "## Leakage fix vs the 3_* reference notebooks\n",
    "The `3_*` notebooks used **Z-scored sheets** (`Z_FFT_*`) whose z-scores were\n",
    "computed across *all* patients including the test fold.  This notebook also runs\n",
    "the raw (non-z-scored) sheet variants where `StandardScaler` — fit only on\n",
    "training data inside the sklearn Pipeline — replaces the pre-computed z-score.\n",
    "Both variants are included for comparison; prefer the raw variants for publication.\n",
    "\n",
    "## CV design\n",
    "- Outer : 10-fold `StratifiedGroupKFold` (patient-level groups, stratified on binary label)\n",
    "- Inner : 5-fold within each outer training set for both Stage 1 and Stage 2\n",
    "- Metric: balanced accuracy (handles the ~50/50 class split)"

In [ ]:
# ── 0. Imports ────────────────────────────────────────────────────────────────
import os, re, glob, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (
    balanced_accuracy_score, f1_score,
    recall_score, roc_auc_score, average_precision_score
)

warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
# ── 1. Configuration ──────────────────────────────────────────────────────────

DATA_DIR = Path('processeddata')
OUT_DIR  = Path('reports/two_stage_leakage_free')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Classification threshold ──────────────────────────────────────────────────
# Class 0: improvers  (Diff <  THRESHOLD)
# Class 1: non-improvers (Diff >= THRESHOLD)
THRESHOLD = -2.0

# ── Cross-validation ──────────────────────────────────────────────────────────
OUTER_N_SPLITS  = 10   # outer folds — each test fold ≈ 10 % of patients
INNER_N_SPLITS  =  5   # inner folds used in both Stage-1 and Stage-2
RANDOM_STATE    = 42

# ── Stage 1 — band selection ──────────────────────────────────────────────────
N_TOP_BANDS = 2        # how many top bands to carry into Stage 2

# ── Stage 2 — LogReg grid ─────────────────────────────────────────────────────
C_VALUES = [1e-3, 1e-2, 1e-1, 1, 10, 100]

PARAM_GRID = [
    {'clf__solver': ['liblinear'], 'clf__penalty': ['l1'], 'clf__C': C_VALUES},
    {'clf__solver': ['liblinear'], 'clf__penalty': ['l2'], 'clf__C': C_VALUES},
    {'clf__solver': ['saga'],      'clf__penalty': ['l1'], 'clf__C': C_VALUES,
     'clf__warm_start': [True]},
    {'clf__solver': ['saga'],      'clf__penalty': ['l2'], 'clf__C': C_VALUES},
]

# ── All band-based EEG sheets ─────────────────────────────────────────────────
# Per-Hz frequency sheets (FFT_abs_1to50Hz_uV2 etc.) are intentionally excluded:
# with 50 individual Hz bins they are too high-dimensional for a ~100-patient cohort.
#
# Leakage note on Z_ sheets:
#   The Z_FFT_* and Z_PeakFreq_Hz sheets were z-scored across ALL patients
#   before the xlsx files were created. That means test-fold patient statistics
#   influenced the z-scores seen by the model — a minor but real leakage.
#   The RAW sheets (no Z_ prefix) with StandardScaler fit only on training data
#   inside the sklearn Pipeline are the leakage-free equivalent.
#   Both are included here so results can be compared; for the final paper
#   prefer the raw-sheet variants.

BANDS_14 = ['Delta', 'Theta', 'Alpha', 'Beta', 'HighBeta', 'Gamma',
             'HighGamma', 'Alpha1', 'Alpha2', 'Beta1', 'Beta2', 'Beta3',
             'Gamma1', 'Gamma2']

BANDS_10 = ['Delta', 'Theta', 'Alpha', 'Beta', 'HighBeta',
            'Alpha1', 'Alpha2', 'Beta1', 'Beta2', 'Beta3']

BANDS_4  = ['Theta', 'Alpha', 'Beta', 'HighBeta']

SHEET_CONFIGS = {
    # ── Raw sheets (leakage-free when paired with StandardScaler in pipeline) ──
    'FFT_abs_bandpower_uV2':    BANDS_14,   # absolute power,  19 ch × 14 bands
    'FFT_rel_bandpower_pct':    BANDS_14,   # relative power,  19 ch × 14 bands
    'PeakFreq_Hz':              BANDS_14,   # peak frequency,  19 ch × 14 bands
    'FFT_Coherence':            BANDS_10,   # coherence,      171 pairs × 10 bands
    'FFT_PhaseLag_PLI':         BANDS_10,   # PLI,            171 pairs × 10 bands
    # ── Z-scored sheets (minor leakage — z-scores computed across all patients)─
    'Z_FFT_abs_bandpower_uV2':  BANDS_10,   # z-scored abs power
    'Z_FFT_rel_bandpower_pct':  BANDS_10,   # z-scored rel power
    'Z_PeakFreq_Hz':            BANDS_4,    # z-scored peak freq (4 bands only)
    'Z_FFT_Coherence':          BANDS_10,   # z-scored coherence
    'Z_FFT_PhaseLag_PLI':       BANDS_10,   # z-scored PLI
}

print('Configuration:')
print(f'  Outer folds   : {OUTER_N_SPLITS}')
print(f'  Inner folds   : {INNER_N_SPLITS}')
print(f'  Top bands     : {N_TOP_BANDS}')
print(f'  Threshold     : Diff < {THRESHOLD} → class 0 (improver)')
print(f'  Sheets ({len(SHEET_CONFIGS)})  :')
for sname, sbands in SHEET_CONFIGS.items():
    tag = '  ← z-scored (minor leakage)' if sname.startswith('Z_') else '  ← raw (leakage-free)'
    print(f'    {sname:<30s}  {len(sbands)} bands{tag}')

In [ ]:
# ── 2. Load outcomes and build patient index ──────────────────────────────────

raw_df = pd.read_excel(DATA_DIR / 'Randomization factors and Primary outcome.xlsx')
pain = (
    raw_df[raw_df['Event Name'].isin(['T1', 'T2'])]
    .pivot_table(index='Patient number', columns='Event Name',
                 values='Pain Unpleasantness')
    .dropna()
)
pain['Diff'] = pain['T2'] - pain['T1']

# EEG file index (one file per patient; keep first if duplicates)
eeg_files = {}
for f in sorted(DATA_DIR.glob('CIPN3*.xlsx')):
    m = re.search(r'CIPN3(\d+)', f.name)
    if m:
        pid = int(m.group(1))
        if pid not in eeg_files:
            eeg_files[pid] = f

# Analysis cohort: patients with both EEG file and valid Diff
valid_pids = pain.index[pain['Diff'].notna()]
cohort_ids = sorted(set(eeg_files) & set(valid_pids))

y_full  = (pain.loc[cohort_ids, 'Diff'].values >= THRESHOLD).astype(int)
groups  = np.array(cohort_ids)   # one integer per patient — used for GroupKFold

n0, n1 = int((y_full == 0).sum()), int((y_full == 1).sum())
print(f'Analysis cohort  : {len(cohort_ids)} patients')
print(f'Class 0 improvers (Diff < {THRESHOLD})  : {n0}')
print(f'Class 1 non-impr  (Diff >= {THRESHOLD}) : {n1}')

In [ ]:
# ── 3. Feature loading utilities ──────────────────────────────────────────────

def load_band_features(xlsx_path: Path, sheet_name: str, bands: list) -> dict:
    """
    Read one patient file and return a dict: band -> np.array([mean, std]).
    mean and std are computed across channels (rows of the sheet).
    Returns an empty dict if the sheet cannot be read.
    """
    try:
        df_s = pd.read_excel(xlsx_path, sheet_name=sheet_name, index_col=0)
    except Exception:
        return {}
    out = {}
    for band in bands:
        if band not in df_s.columns:
            continue
        vals = df_s[band].values.astype(float)
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            continue
        out[band] = np.array([vals.mean(), vals.std()], dtype=float)
    return out


def build_X_for_bands(cache: list, row_idx, bands: list):
    """
    Build feature matrix from a pre-loaded cache.

    Parameters
    ----------
    cache    : list of dicts, one per patient (band -> [mean, std])
    row_idx  : array-like of integer positions into cache
    bands    : list of band names to include

    Returns None if any patient is missing a requested band.
    """
    rows = []
    for i in row_idx:
        d = cache[i]
        if any(b not in d for b in bands):
            return None
        rows.append(np.concatenate([d[b] for b in bands]))
    return np.vstack(rows)


print('Feature utilities defined.')

In [ ]:
# ── 4. Pre-load feature caches for every sheet ────────────────────────────────
# Each entry in `all_caches[sheet_name]` is a list of dicts (one per patient).
# Loading upfront once is fast and avoids repeated Excel reads inside the CV loop.

all_caches = {}
for sheet_name, bands in SHEET_CONFIGS.items():
    print(f'\nLoading sheet: {sheet_name}')
    cache = []
    ok = 0
    for pid in cohort_ids:
        d = load_band_features(eeg_files[pid], sheet_name, bands)
        cache.append(d)
        if d:
            ok += 1
    all_caches[sheet_name] = cache
    print(f'  Loaded {ok}/{len(cohort_ids)} patients  |  '
          f'Example bands: {list(cache[0].keys())[:5]}')

print('\nAll caches ready.')

In [ ]:
# ── 5. Save outer fold assignments to CSV ─────────────────────────────────────
# Creates a reproducible record of which patients land in which test fold.

outer_cv = StratifiedGroupKFold(
    n_splits=OUTER_N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

X_dummy = np.zeros((len(y_full), 1))
outer_splits = list(outer_cv.split(X_dummy, y_full, groups=groups))

fold_assign_rows = []
for fold_i, (tr, te) in enumerate(outer_splits, 1):
    for idx in te:
        fold_assign_rows.append({
            'patient_id': cohort_ids[idx],
            'outer_fold': fold_i,
            'split':      'test',
            'label':      int(y_full[idx]),
        })
    for idx in tr:
        fold_assign_rows.append({
            'patient_id': cohort_ids[idx],
            'outer_fold': fold_i,
            'split':      'train',
            'label':      int(y_full[idx]),
        })

fold_df = pd.DataFrame(fold_assign_rows)
fold_csv = OUT_DIR / 'fold_assignments.csv'
fold_df.to_csv(fold_csv, index=False)
print(f'Fold assignments saved → {fold_csv}')

# Quick sanity: no patient in both train and test for any fold
for fold_i, (tr, te) in enumerate(outer_splits, 1):
    overlap = set(groups[tr]) & set(groups[te])
    assert len(overlap) == 0, f'Fold {fold_i}: patient overlap detected!'

print(f'\nOuter CV: {OUTER_N_SPLITS} folds — no patient leakage confirmed.')
print(fold_df.groupby('outer_fold')['split'].value_counts().unstack().to_string())

In [ ]:
# ── 6. Helper: make simple per-band pipeline ──────────────────────────────────

def make_base_pipe():
    """StandardScaler + LogReg with balanced class weights."""
    return Pipeline([
        ('scl', StandardScaler()),
        ('clf', LogisticRegression(
            C=1.0, class_weight='balanced',
            max_iter=10000, tol=1e-3, random_state=RANDOM_STATE)),
    ])


def make_grid_pipe():
    """Pipeline used as the estimator inside GridSearchCV."""
    return Pipeline([
        ('scl', StandardScaler()),
        ('clf', LogisticRegression(
            class_weight='balanced',
            max_iter=50000, tol=1e-3, random_state=RANDOM_STATE)),
    ])


print('Pipeline helpers defined.')

In [ ]:
# ── 7. Main two-stage CV loop — runs for every sheet ─────────────────────────

def mean_std_str(arr):
    return f'{np.mean(arr):.3f}±{np.std(arr):.3f}'


all_sheet_results = {}   # sheet_name -> list of per-fold dicts

for sheet_name, bands in SHEET_CONFIGS.items():
    print('\n' + '='*80)
    print(f'SHEET: {sheet_name}  ({len(bands)} candidate bands)')
    print('='*80)

    cache = all_caches[sheet_name]
    fold_results = []

    # ── aggregate band selection tallies across all outer folds ───────────────
    global_band_wins  = Counter()   # wins across ALL outer folds
    global_band_count = Counter()   # inner folds attempted per band

    for fold_i, (tr, te) in enumerate(outer_splits, 1):
        g_tr, g_te = groups[tr], groups[te]
        y_tr, y_te = y_full[tr], y_full[te]

        # Slice caches by fold membership
        cache_idx_tr = list(tr)   # index positions into `cache` for training patients
        cache_idx_te = list(te)

        print(f'\n[Fold {fold_i}/{OUTER_N_SPLITS}]  '
              f'train={len(tr)} patients  test={len(te)} patients  '
              f'(cls0_tr={int((y_tr==0).sum())} cls1_tr={int((y_tr==1).sum())})')

        # ── Stage 1: Band selection via inner-fold win counting ────────────────
        inner_cv = StratifiedGroupKFold(
            n_splits=INNER_N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

        # Dummy X for inner splits (we need only the indices)
        X_dummy_tr = np.zeros((len(tr), 1))
        inner_splits = list(inner_cv.split(X_dummy_tr, y_tr, groups=g_tr))

        fold_band_wins  = Counter()
        fold_band_scores = {b: [] for b in bands}  # inner-fold bal_acc per band

        for inner_i, (itr_rel, ival_rel) in enumerate(inner_splits, 1):
            # Map relative inner indices → absolute cache positions
            itr_abs  = [cache_idx_tr[k] for k in itr_rel]
            ival_abs = [cache_idx_tr[k] for k in ival_rel]
            y_itr    = y_tr[itr_rel]
            y_ival   = y_tr[ival_rel]

            best_band_score = -1.0
            best_band_name  = None

            for band in bands:
                X_itr  = build_X_for_bands(cache, itr_abs,  [band])
                X_ival = build_X_for_bands(cache, ival_abs, [band])
                if X_itr is None or X_ival is None:
                    continue

                pipe = make_base_pipe()
                try:
                    pipe.fit(X_itr, y_itr)
                    ba = balanced_accuracy_score(
                        y_ival, pipe.predict(X_ival))
                except Exception:
                    continue

                fold_band_scores[band].append(ba)
                global_band_count[band] += 1

                if ba > best_band_score:
                    best_band_score = ba
                    best_band_name  = band

            if best_band_name is not None:
                fold_band_wins[best_band_name] += 1
                global_band_wins[best_band_name] += 1

        # Average inner-fold score per band (for logging)
        avg_scores = {b: float(np.mean(v)) if v else 0.0
                      for b, v in fold_band_scores.items()}

        # Select top-N bands by win count;
        # break ties with average inner-fold balanced accuracy
        ranked_bands = sorted(
            bands,
            key=lambda b: (fold_band_wins[b], avg_scores[b]),
            reverse=True
        )
        selected_bands = ranked_bands[:N_TOP_BANDS]

        print(f'  Stage 1 band wins ({INNER_N_SPLITS} inner folds):')
        for b in bands:
            marker = ' ◀ SELECTED' if b in selected_bands else ''
            print(f'    {b:<12s}  wins={fold_band_wins[b]}/{INNER_N_SPLITS}  '
                  f'avg_inner_BA={avg_scores[b]:.3f}{marker}')
        print(f'  → Selected bands: {selected_bands}')

        # ── Stage 2: Grid search with selected bands ───────────────────────────
        X_tr_sel = build_X_for_bands(cache, cache_idx_tr, selected_bands)
        X_te_sel = build_X_for_bands(cache, cache_idx_te, selected_bands)

        if X_tr_sel is None or X_te_sel is None:
            print(f'  [WARN] Missing band data for fold {fold_i}, skipping.')
            continue

        inner_cv2 = StratifiedGroupKFold(
            n_splits=INNER_N_SPLITS, shuffle=True,
            random_state=RANDOM_STATE + fold_i)

        gs = GridSearchCV(
            make_grid_pipe(),
            param_grid=PARAM_GRID,
            scoring='balanced_accuracy',
            cv=inner_cv2,
            n_jobs=-1, refit=True, verbose=0
        )
        gs.fit(X_tr_sel, y_tr, groups=g_tr)

        print(f'  Stage 2 best params : {gs.best_params_}')
        print(f'  Stage 2 inner BA    : {gs.best_score_:.3f}')

        # ── Evaluate on test fold ──────────────────────────────────────────────
        best_est = gs.best_estimator_
        p_te  = best_est.predict_proba(X_te_sel)[:, 1]
        yh_te = (p_te >= 0.5).astype(int)

        ba_te   = balanced_accuracy_score(y_te, yh_te)
        f1_te   = f1_score(y_te, yh_te, zero_division=0)
        rec0_te = recall_score(y_te, yh_te, pos_label=0, zero_division=0)  # specificity
        rec1_te = recall_score(y_te, yh_te, pos_label=1, zero_division=0)  # sensitivity

        # AUC / AP (only meaningful when both classes present in test)
        try:
            auc_te = roc_auc_score(y_te, p_te)
            ap_te  = average_precision_score(y_te, p_te)
        except ValueError:
            auc_te = float('nan')
            ap_te  = float('nan')

        print(f'  Test fold BA={ba_te:.3f}  AUC={auc_te:.3f}  '
              f'F1={f1_te:.3f}  Sens={rec1_te:.3f}  Spec={rec0_te:.3f}')

        fold_results.append({
            'fold':            fold_i,
            'sheet':           sheet_name,
            'n_train':         len(tr),
            'n_test':          len(te),
            'selected_bands':  selected_bands,
            'band_wins':       dict(fold_band_wins),
            'avg_inner_ba':    avg_scores,
            'inner_ba_stage2': gs.best_score_,
            'best_params':     gs.best_params_,
            'ba':    ba_te,
            'auc':   auc_te,
            'ap':    ap_te,
            'f1':    f1_te,
            'sensitivity': rec1_te,
            'specificity': rec0_te,
        })

    # ── Sheet-level summary ────────────────────────────────────────────────────
    all_sheet_results[sheet_name] = fold_results

    bas  = [r['ba']  for r in fold_results]
    aucs = [r['auc'] for r in fold_results if not np.isnan(r['auc'])]
    f1s  = [r['f1']  for r in fold_results]
    sens = [r['sensitivity'] for r in fold_results]
    spec = [r['specificity'] for r in fold_results]

    print(f'\n{"─"*60}')
    print(f'[{sheet_name}] {OUTER_N_SPLITS}-fold summary')
    print(f'  BalAcc : {mean_std_str(bas)}')
    print(f'  AUC    : {mean_std_str(aucs)}')
    print(f'  F1     : {mean_std_str(f1s)}')
    print(f'  Sens   : {mean_std_str(sens)}')
    print(f'  Spec   : {mean_std_str(spec)}')
    print(f'\n  Global band wins across all {OUTER_N_SPLITS * INNER_N_SPLITS} inner folds:')
    for b in bands:
        total = OUTER_N_SPLITS * INNER_N_SPLITS
        print(f'    {b:<12s}  {global_band_wins[b]:>3}/{total}')

In [ ]:
# ── 8. Results summary across all sheets ──────────────────────────────────────

print('=' * 70)
print(f'{"Sheet":<30s}  {"BalAcc":>12s}  {"AUC":>12s}  {"F1":>10s}')
print('-' * 70)

for sheet_name, fold_results in all_sheet_results.items():
    bas  = [r['ba']  for r in fold_results]
    aucs = [r['auc'] for r in fold_results if not np.isnan(r['auc'])]
    f1s  = [r['f1']  for r in fold_results]
    print(f'{sheet_name:<30s}  '
          f'{mean_std_str(bas):>12s}  '
          f'{mean_std_str(aucs):>12s}  '
          f'{mean_std_str(f1s):>10s}')

print('-' * 70)
print('Chance: 0.500')

In [ ]:
# ── 9. Band selection frequency table ─────────────────────────────────────────
# For each sheet: how often was each band selected as one of the top-N across outer folds?

print('Band selection frequency (how many outer folds each band was selected in):')
print(f'  (N_TOP_BANDS={N_TOP_BANDS}, OUTER_N_SPLITS={OUTER_N_SPLITS})')

band_sel_records = []
for sheet_name, fold_results in all_sheet_results.items():
    bands = SHEET_CONFIGS[sheet_name]
    sel_counts = Counter()
    for r in fold_results:
        for b in r['selected_bands']:
            sel_counts[b] += 1

    print(f'\n  {sheet_name}:')
    for b in bands:
        bar = '█' * sel_counts[b] + '░' * (OUTER_N_SPLITS - sel_counts[b])
        print(f'    {b:<12s}  {bar}  {sel_counts[b]}/{OUTER_N_SPLITS}')

    for b in bands:
        band_sel_records.append({
            'sheet': sheet_name,
            'band':  b,
            'times_selected': sel_counts[b],
            'out_of': OUTER_N_SPLITS,
        })

band_sel_df = pd.DataFrame(band_sel_records)
band_sel_df.to_csv(OUT_DIR / 'band_selection_frequency.csv', index=False)
print(f'\nBand selection frequency saved → {OUT_DIR}/band_selection_frequency.csv')

In [ ]:
# ── 10. Export per-fold results to CSV ────────────────────────────────────────

all_rows = []
for sheet_name, fold_results in all_sheet_results.items():
    for r in fold_results:
        row = {
            'sheet':            r['sheet'],
            'fold':             r['fold'],
            'n_train':          r['n_train'],
            'n_test':           r['n_test'],
            'selected_bands':   ' | '.join(r['selected_bands']),
            'inner_ba_stage2':  r['inner_ba_stage2'],
            'best_solver':      r['best_params'].get('clf__solver', ''),
            'best_penalty':     r['best_params'].get('clf__penalty', ''),
            'best_C':           r['best_params'].get('clf__C', ''),
            'test_ba':          r['ba'],
            'test_auc':         r['auc'],
            'test_ap':          r['ap'],
            'test_f1':          r['f1'],
            'test_sensitivity': r['sensitivity'],
            'test_specificity': r['specificity'],
        }
        all_rows.append(row)

results_df = pd.DataFrame(all_rows)
results_csv = OUT_DIR / 'cv_results.csv'
results_df.to_csv(results_csv, index=False)

print(results_df.to_string(index=False))
print(f'\nResults saved → {results_csv}')

In [ ]:
# ── 11. Optional: per-fold bar chart ─────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(SHEET_CONFIGS),
                          figsize=(5 * len(SHEET_CONFIGS), 4.5),
                          sharey=True)
if len(SHEET_CONFIGS) == 1:
    axes = [axes]

colors = ['#2196F3', '#FF9800', '#4CAF50']

for ax, (sheet_name, fold_results), col in zip(
        axes, all_sheet_results.items(), colors):
    folds = [r['fold'] for r in fold_results]
    bas   = [r['ba']   for r in fold_results]
    ax.bar(folds, bas, color=col, alpha=0.8, edgecolor='white')
    ax.axhline(0.5,           ls='--', color='gray', lw=1.2, label='Chance')
    ax.axhline(np.mean(bas),  ls='-',  color=col,   lw=2.0, alpha=0.6,
               label=f'Mean={np.mean(bas):.3f}')
    ax.set_xlabel('Outer fold', fontsize=11)
    ax.set_ylabel('Balanced Accuracy', fontsize=11)
    ax.set_title(sheet_name.replace('_', '\n'), fontsize=10)
    ax.set_xticks(folds)
    ax.set_ylim(0.2, 1.05)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Two-Stage Leakage-Free Pipeline — Per-Fold BalAcc',
             fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(OUT_DIR / 'per_fold_balacc.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Figure saved → {OUT_DIR}/per_fold_balacc.png')